# Stage1 HYG−LQD 信用风险偏好 20 日来源

冻结原始 close+当日除息金额构造两只 ETF 的单期总收益，取过去20个源交易日 HYG−LQD 对数累计差；源日收盘只可供下一 Stage1 决策。全 2011–2022 训练和复用 2023–2026 评价日保留，不读 SOXX 标签、不拟合、不建账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex,math
from pathlib import Path
from datetime import datetime,timezone
from zoneinfo import ZoneInfo
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_credit_risk_pair_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
s=plan['source'];assert sha(root/s['raw_price_result'])==s['raw_price_result_sha256'] and sha(root/s['event_result'])==s['event_result_sha256']
prices_state=json.loads((root/s['raw_price_result']).read_text());event_state=json.loads((root/s['event_result']).read_text())
assert prices_state['status']=='HYG_LQD_current_chart_raw_full_span_source_received_PIT_pending' and event_state['status']=='HYG_LQD_retrospective_dividend_event_sources_qualified_for_full_span_PIT_pending'
assert prices_state['NAS_remote_and_scoped_restore_verified'] and event_state['NAS_remote_and_scoped_restore_verified']
state={'status':'started','raw_price_JSON_reads':0,'event_JSON_reads':0,'Qlib_D_calendar_calls':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
raw=root/s['raw_root'];frames=[];quality=[]
for symbol in s['symbols']:
 price_receipt=next(x for x in prices_state['checks'] if x['symbol']==symbol);event_receipt=next(x for x in event_state['checks'] if x['symbol']==symbol)
 price_path=raw/(symbol+'.json');event_path=raw/'events'/(symbol+'.json');assert sha(price_path)==price_receipt['raw_sha256'] and sha(event_path)==event_receipt['raw_sha256']
 price_payload=json.loads(price_path.read_text());event_payload=json.loads(event_path.read_text());state['raw_price_JSON_reads']+=1;state['event_JSON_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 z=price_payload['chart']['result'][0];q=event_payload['chart']['result'][0];assert z['meta']['symbol']==q['meta']['symbol']==symbol and z['timestamp']==q['timestamp'] and z['meta']['exchangeTimezoneName']=='America/New_York'
 dates=pd.DatetimeIndex([datetime.fromtimestamp(t,timezone.utc).astimezone(ZoneInfo('America/New_York')).date() for t in z['timestamp']]);assert dates.is_unique and dates.is_monotonic_increasing and len(dates)==4227
 close=np.asarray(z['indicators']['quote'][0]['close'],dtype=float);adj=np.asarray(z['indicators']['adjclose'][0]['adjclose'],dtype=float);assert len(close)==len(adj)==len(dates) and np.isfinite(close).all() and (close>0).all()
 events=q.get('events') or {};assert len(events.get('splits') or {})==0;divevents=events.get('dividends') or {};assert len(divevents)==200
 dividends=pd.Series(0.0,index=dates)
 for key,value in divevents.items():
  day=pd.Timestamp(datetime.fromtimestamp(int(key),timezone.utc).astimezone(ZoneInfo('America/New_York')).date());amount=float(value['amount']);assert day in dividends.index and math.isfinite(amount) and amount>0
  dividends.loc[day]+=amount
 gross=pd.Series((close+dividends.to_numpy(float))/pd.Series(close).shift(1).to_numpy(float),index=dates);gross.iloc[0]=np.nan
 assert np.isfinite(gross.iloc[1:]).all() and gross.iloc[1:].gt(0).all()
 logret=np.log(gross);rolling20=logret.rolling(20,min_periods=20).sum()
 adjgross=pd.Series(adj,index=dates)/pd.Series(adj,index=dates).shift(1);delta=(np.log(adjgross)-logret).abs().dropna()
 frames.append(pd.DataFrame({'signal_date':dates,symbol+'_raw_close':close,symbol+'_dividend':dividends.to_numpy(float),symbol+'_log_total_return':logret.to_numpy(float),symbol+'_log_total_return_20':rolling20.to_numpy(float)}).set_index('signal_date'))
 quality.append({'symbol':symbol,'price_rows':len(dates),'dividend_events':len(divevents),'split_events':0,'first_date':str(dates.min().date()),'last_date':str(dates.max().date()),'adjclose_audit_median_abs_logreturn_gap':float(delta.median()),'adjclose_audit_max_abs_logreturn_gap':float(delta.max()),'adjclose_audit_days_abs_gap_gt_1bp':int((delta>.0001).sum()),'raw_price_sha256':price_receipt['raw_sha256'],'event_sha256':event_receipt['raw_sha256']})
assert frames[0].index.equals(frames[1].index)
source=pd.concat(frames,axis=1);source['HYG_minus_LQD_log_total_return_20']=source['HYG_log_total_return_20']-source['LQD_log_total_return_20']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['train_signal_span'][0],end_time=plan['held_reused_signal_span'][1],freq='day'));state['Qlib_D_calendar_calls']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
train=calendar[(calendar>=pd.Timestamp(plan['train_signal_span'][0]))&(calendar<=pd.Timestamp(plan['train_signal_span'][1]))];held=calendar[(calendar>=pd.Timestamp(plan['held_reused_signal_span'][0]))&(calendar<=pd.Timestamp(plan['held_reused_signal_span'][1]))]
assert len(train)==3020 and len(held)==944 and len(calendar)==3964
mapped=source.reindex(calendar);mapped['source_known']=mapped.HYG_minus_LQD_log_total_return_20.notna()
assert mapped.source_known.all() and np.isfinite(mapped.HYG_minus_LQD_log_total_return_20).all()
mapped['historical_adjustment_vintage_qualified']=False
static=mapped[['HYG_minus_LQD_log_total_return_20','source_known','historical_adjustment_vintage_qualified']].copy()
static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument'])
static=pd.concat({'feature':static[['HYG_minus_LQD_log_total_return_20']],'meta':static[['source_known','historical_adjustment_vintage_qualified']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['train_signal_span']),'held':tuple(plan['held_reused_signal_span'])})
assert len(dataset.prepare('train',col_set=['feature','meta']))==3020 and len(dataset.prepare('held',col_set=['feature','meta']))==944
source_audit={'status':'HYG_LQD_raw_close_plus_dividend_20_session_source_calendar_complete_PIT_vintage_pending','plan_sha256':sha(card/'plan.json'),'quality':quality,'train_Qlib_sessions':len(train),'held_Qlib_sessions':len(held),'known_train_sessions':int(mapped.source_known.loc[train].sum()),'known_held_sessions':int(mapped.source_known.loc[held].sum()),'feature_train_min':float(mapped.loc[train,'HYG_minus_LQD_log_total_return_20'].min()),'feature_train_max':float(mapped.loc[train,'HYG_minus_LQD_log_total_return_20'].max()),'historical_adjustment_vintage_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(source_audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('train',col_set=['feature','meta']))==3020
tracking=card/'qlib_recorder'
with R.start(experiment_name='Stage1_HYG_minus_LQD_20_source',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'prices':s['raw_price_result_sha256'],'events':s['event_result_sha256']})
 rec.save_objects(artifact_path='analysis',raw_source_daily=source,mapped_signals=mapped,source_audit=source_audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_train_days=len(train),source_held_days=len(held),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/mapped_signals'))==3964 and len(checked.load_object('metadata/dataset').prepare('train',col_set=['feature','meta']))==3020
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'Stage1_HYG_minus_LQD_20_source','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='credit-pair-source-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source_tar:names=[m.name for m in source_tar.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'HYG_minus_LQD_20_session_source_calendar_complete_historical_vintage_unqualified_no_fit','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'train':source_audit['known_train_sessions'],'held':source_audit['known_held_sessions'],'quality':quality,'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'train': 3020, 'held': 944, 'quality': [{'symbol': 'HYG', 'price_rows': 4227, 'dividend_events': 200, 'split_events': 0, 'first_date': '2010-01-04', 'last_date': '2026-10-22', 'adjclose_audit_median_abs_logreturn_gap': 2.224808769259244e-07, 'adjclose_audit_max_abs_logreturn_gap': 0.00014102025526735104, 'adjclose_audit_days_abs_gap_gt_1bp': 2, 'raw_price_sha256': 'bd9e406c67268b0f5582dc3ca2d3371d56d6d3fee9d8be11a66d555f0062e0fe', 'event_sha256': '6259b766f7b89c47af1ffebd1e6bdc1d3aabc39c81c81b8690be95e3b9a280f6'}, {'symbol': 'LQD', 'price_rows': 4227, 'dividend_events': 200, 'split_events': 0, 'first_date': '2010-01-04', 'last_date': '2026-10-22', 'adjclose_audit_median_abs_logreturn_gap': 1.9613003273593124e-07, 'adjclose_audit_max_abs_logreturn_gap': 5.968809224559876e-05, 'adjclose_audit_days_abs_gap_gt_1bp': 0, 'raw_price_sha256': 'b8db8de160b610b4c2a3021fa371acd5f68adcf3d1e616854d2adb04a50ca582', 'event_sha256': '27166ce97e6b2e281eaa35d00ffd8a003c6a89bacd8cef9cb54c8c7215d8da7a'}]

## 中文结论

HYG/LQD 原始 close+200 条各自除息分红构成已结束的 20 日相对总收益；原 Stage1 Qlib 训练 3,020/3,020、复用评价 944/944 日均有值。当前回溯 adjclose 仅用于诊断、不作输入；历史首次分红版本仍未认证。DatasetH/Recorder `018a98da05724acda3c50b11a7b9a69f` 与 NAS SHA-256 `0aa49fe9ef45ed6d24bf1df9e1e38de8bdfb155231050591370a47cf8b12f243` 回读通过。0 SOXX 结果、0 拟合、0 账户。下一步仅作冻结的匹配信息检验，不能直接晋升。